# IMDB Movie Reviews — Binary Text Classification

**Daily Challenge — Week 5 / Day 5**

Classify IMDB movie reviews as **positive (1)** or **negative (0)** with a small
feedforward network. Reviews come pre-encoded as integer sequences (each integer
is a word rank); we one-hot encode them into 10,000-dim multi-hot vectors and feed
a stack of `Dense` layers.

**Steps**
1. Preprocess the data (load, vectorize, split)
2. Build the model
3. Train (20 epochs, batch 512)
4. Evaluate (curves, retrain at optimal epochs, test)
5. Analyze results

## 1. Preprocess the data

The IMDB set ships split 25,000 train / 25,000 test, each balanced 50/50. Keeping
`num_words=10000` restricts every review to the 10k most frequent tokens (rarer
words are dropped), which bounds our vector width.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

NUM_WORDS = 10_000  # keep the 10k most frequent tokens

(train_data, train_labels), (test_data, test_labels) = \
    keras.datasets.imdb.load_data(num_words=NUM_WORDS)

print('train sequences:', len(train_data), '| test sequences:', len(test_data))
print('example review (first 12 token ids):', train_data[0][:12])
print('label of first review:', train_labels[0], '(1=positive, 0=negative)')
print('max token id across train:', max(max(seq) for seq in train_data))

### Why we must vectorize

A review is a *variable-length list of integers* — a network needs fixed-size
numeric tensors. We **one-hot (multi-hot) encode**: each review becomes a
10,000-dim vector that is `1` at the indices of words it contains and `0`
everywhere else. `[3, 5] -> [0,0,0,1,0,1,0,...]`. This discards word order and
counts but is enough for strong sentiment classification and lets the first
`Dense` layer consume floating-point vectors directly.

In [ ]:
def vectorize_sequences(sequences, dimension=NUM_WORDS):
    # All-zero matrix of shape (len(sequences), dimension)
    results = np.zeros((len(sequences), dimension), dtype='float32')
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0   # set the indices that appear to 1
    return results

x_train = vectorize_sequences(train_data)
x_test  = vectorize_sequences(test_data)

# Labels to float32 vectors
y_train = np.asarray(train_labels).astype('float32')
y_test  = np.asarray(test_labels).astype('float32')

print('x_train shape:', x_train.shape, '| x_test shape:', x_test.shape)
print('one review is now a vector of', x_train.shape[1], 'values; sum (unique words):', int(x_train[0].sum()))

### Train / validation / test split

The test set (25,000) is held out and **only** touched for the final evaluation.
We carve a validation set out of the training data — the first 10,000 reviews —
to monitor generalisation during training. That leaves 15,000 for training.

In [ ]:
x_val,        partial_x_train = x_train[:10000], x_train[10000:]
y_val,        partial_y_train = y_train[:10000], y_train[10000:]

print('train:', partial_x_train.shape[0],
      '| val:', x_val.shape[0],
      '| test:', x_test.shape[0])

## 2. Build the model

Input is a plain 10,000-dim vector and the label is a scalar 0/1 — the simplest
setup there is. A small stack of fully-connected `Dense` layers with `relu`
activations does well. Two hidden layers of 16 units give enough capacity without
overfitting a bag-of-words representation, and the final **`Dense(1, sigmoid)`**
outputs P(positive).

**Loss & optimizer:** since the output is a probability for a binary target,
`binary_crossentropy` is the right loss (it measures the distance between predicted
and true distributions, which suits probabilities better than MSE here). We use
**RMSprop** as the optimizer and track **accuracy**.

In [ ]:
from tensorflow.keras import layers, models

def build_model():
    model = models.Sequential([
        layers.Input((NUM_WORDS,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1,  activation='sigmoid'),
    ], name='imdb_mlp')
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

build_model().summary()

## 3. Train the model

Train for **20 epochs** with **batch size 512**, monitoring the validation set
each epoch. We deliberately over-train here so the curves in Step 4 reveal where
overfitting begins.

In [ ]:
model = build_model()
history = model.fit(
    partial_x_train, partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
)

## 4. Evaluate the model

Plot training vs validation loss and accuracy. The tell-tale sign of overfitting:
**training loss keeps falling while validation loss bottoms out and turns back
up** (and validation accuracy plateaus). The epoch where validation loss is
minimal is the point to stop.

In [ ]:
h = history.history
epochs = range(1, len(h['loss']) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, h['loss'],     'bo-', label='train loss')
ax[0].plot(epochs, h['val_loss'], 'r^-', label='val loss')
ax[0].set_title('Training vs validation loss')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend()

ax[1].plot(epochs, h['accuracy'],     'bo-', label='train acc')
ax[1].plot(epochs, h['val_accuracy'], 'r^-', label='val acc')
ax[1].set_title('Training vs validation accuracy')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend()
plt.tight_layout(); plt.show()

best_epoch = int(np.argmin(h['val_loss'])) + 1
print('Validation loss is lowest at epoch:', best_epoch)

### Retrain at the optimal number of epochs

Validation loss typically bottoms out around epoch **3–4** for this model; past
that the network starts memorising training quirks. We rebuild a fresh model and
train only to that point (using the detected `best_epoch`) to get the best
generalising weights, then evaluate on the untouched test set.

In [ ]:
final_model = build_model()
final_model.fit(partial_x_train, partial_y_train,
                epochs=best_epoch, batch_size=512,
                validation_data=(x_val, y_val), verbose=1)

test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}')

## 5. Analyze the results

**Training vs validation behaviour.** Training loss decreases monotonically across
all 20 epochs and training accuracy approaches ~100% — the model has more than
enough capacity to fit the training data. Validation loss, however, improves only
for the first few epochs, reaches its minimum around epoch 3–4, and then climbs
steadily while validation accuracy flattens near ~88%. That growing gap between
the train and validation curves is the signature of **overfitting**: beyond the
early epochs the network is memorising training-specific noise rather than learning
generalisable sentiment cues.

**Mitigation applied.** Rather than train for the full 20 epochs, we retrained a
fresh model for only `best_epoch` epochs (the validation-loss minimum), which is
the simplest form of early stopping. Further regularisation could include dropout,
L2 weight decay, or a smaller network.

**Final test performance.** The retrained model reaches roughly **~88% test
accuracy** with a low binary cross-entropy loss — strong for a bag-of-words model
that ignores word order. (Exact numbers print from the cell above when you run it.)

**Takeaways:** one-hot/bag-of-words + a tiny MLP is a solid sentiment baseline; the
main risk is over-training, which the validation curve makes obvious and early
stopping fixes. To push higher you'd move to learned word embeddings + sequence
models (Embedding → Conv1D/LSTM) that exploit word order.